# Module 3 — Exercises (Student Worksheet)
### Attention, Transformers & the Particle Transformer (ParT) · TIFR ML School 2026

Student worksheet for the eight exercises at the end of
`Module3_Attention_Transformers_ParT.ipynb`. **Self-contained**: the *Setup* re-defines scaled-dot-product /
multi-head attention, the encoder block, attention pooling, ParT's pairwise bias, and loads JetClass as
masked sets — then trains a **base plain Transformer** and a **base ParT** that the exercises reuse.

| # | Exercise | Idea it drives home |
|---|----------|---------------------|
| 1 | **Ablate the bias** | which of ParT's 4 pairwise features carries the advantage |
| 2 | **Depth / heads / width** | where capacity stops paying off for jets |
| 3 | **Class-attn vs mean-pool** | how much learnable pooling actually buys |
| 4 | **ISAB trade-off** | the AUC you pay for `O(N²)→O(Nm)` linear attention |
| 5 | **Attention → graph** | does attention rediscover ParticleNet's k-NN neighbours? |
| 6 | **Three-way race** | PFN vs ParticleNet vs ParT on one ROC, same jets |
| 7 | **Boost augmentation** | inputs aren't boost-invariant; ParT's bias already is |
| 8 | **Faithful attribution** | gradient-weighted rollout (Chefer et al.) vs plain rollout |

> **Design choices that keep this fast & clean:** the `ParticleTransformer` is **configurable**
> (`use_pair_bias`, `pair_feat_idx`, `pool`), so exercises 1 & 3 are just constructor flags; a single base
> plain/ParT pair is trained once and reused for interpretability (5, 8) and the race (6).
>
> **Speed knobs:** `N_PER_CLS`, `DIM`, `DEPTH`, `EPOCHS`. Defaults are deliberately small so all ~20 trainings
> finish in a few minutes on a laptop; raise them to approach the module's headline AUC (ParT ≈ 0.99).

> **How to use this worksheet.** Run the **Setup** section as-is (it loads the data and provides the models/helpers from the module). Then complete each exercise where you see `# TODO` and `raise NotImplementedError`. Read the **prompt** above each exercise — it sketches the approach. Check your work against the matching `*_Exercises_Solutions.ipynb`.

## Setup — attention machinery, ParT, data, base models

In [ ]:
import os, math, time, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

torch.manual_seed(0); np.random.seed(0)
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("torch", torch.__version__, "| device", device)

In [ ]:
# --- attention primitives (from Module 3) -----------------------------------------------
def scaled_dot_product_attention(q, k, v, key_mask=None, attn_bias=None):
    d = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d)
    if attn_bias is not None: scores = scores + attn_bias
    if key_mask is not None:  scores = scores.masked_fill(~key_mask.unsqueeze(-2).bool(), float("-inf"))
    attn = torch.nan_to_num(scores.softmax(dim=-1))
    return attn @ v, attn

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); assert dim % heads == 0
        self.h, self.dk = heads, dim // heads
        self.qkv = nn.Linear(dim, 3*dim); self.proj = nn.Linear(dim, dim)
    def forward(self, x, key_mask, attn_bias=None):
        B, N, _ = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        out, attn = scaled_dot_product_attention(qkv[0], qkv[1], qkv[2],
            key_mask=key_mask[:, None, :] if key_mask is not None else None, attn_bias=attn_bias)
        return self.proj(out.transpose(1, 2).reshape(B, N, -1)), attn

class EncoderBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4, drop=0.1):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_ratio*dim), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(mlp_ratio*dim, dim))
        self.drop = nn.Dropout(drop)
    def forward(self, x, key_mask, attn_bias=None):
        a, attn = self.attn(self.ln1(x), key_mask, attn_bias)
        x = x + self.drop(a); x = x + self.mlp(self.ln2(x))
        return x, attn

class AttentionPooling(nn.Module):
    """Set Transformer PMA: one learnable class query attends over particles -> one jet vector."""
    def __init__(self, dim, heads):
        super().__init__(); self.h, self.dk = heads, dim // heads
        self.cls = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.q, self.kv, self.proj = nn.Linear(dim, dim), nn.Linear(dim, 2*dim), nn.Linear(dim, dim)
        self.last_attn = None
    def forward(self, x, key_mask):
        B, N, _ = x.shape
        q = self.q(self.cls.expand(B, 1, -1)).reshape(B, 1, self.h, self.dk).transpose(1, 2)
        kv = self.kv(x).reshape(B, N, 2, self.h, self.dk).permute(2, 0, 3, 1, 4)
        out, self.last_attn = scaled_dot_product_attention(q, kv[0], kv[1], key_mask=key_mask[:, None, :])
        return self.proj(out.transpose(1, 2).reshape(B, 1, -1)).squeeze(1)

In [ ]:
# --- ParT pairwise bias + CONFIGURABLE Particle Transformer ------------------------------
PAIR_NAMES = [r"$\ln\Delta$", r"$\ln k_T$", r"$\ln z$", r"$\ln m^2$"]

def pairwise_features(p4, eps=1e-8):
    """p4:(B,N,4)=(px,py,pz,E) -> U:(B,N,N,4)=[lnDelta, ln kT, ln z, ln m^2]."""
    px, py, pz, E = p4[..., 0], p4[..., 1], p4[..., 2], p4[..., 3]
    pt  = torch.sqrt(px**2 + py**2).clamp(min=1e-6); phi = torch.atan2(py, px)
    y   = 0.5 * torch.log(((E + pz).clamp(min=eps)) / ((E - pz).clamp(min=eps)))
    dy   = y[:, :, None] - y[:, None, :]
    dphi = phi[:, :, None] - phi[:, None, :]; dphi = torch.atan2(torch.sin(dphi), torch.cos(dphi))
    delta = torch.sqrt(dy**2 + dphi**2).clamp(min=1e-6)
    ptmin = torch.minimum(pt[:, :, None], pt[:, None, :])
    kt = (ptmin * delta).clamp(min=1e-6)
    z  = (ptmin / (pt[:, :, None] + pt[:, None, :] + eps)).clamp(min=1e-6, max=1.0)
    sE, spx = E[:, :, None]+E[:, None, :], px[:, :, None]+px[:, None, :]
    spy, spz = py[:, :, None]+py[:, None, :], pz[:, :, None]+pz[:, None, :]
    m2 = (sE**2 - spx**2 - spy**2 - spz**2).clamp(min=1e-6)
    return torch.stack([torch.log(delta), torch.log(kt), torch.log(z), torch.log(m2)], dim=-1)

class ParticleTransformer(nn.Module):
    """Configurable: use_pair_bias, pair_feat_idx (subset of the 4 features), pool in {'attn','mean'}."""
    def __init__(self, in_feat, n_classes=2, dim=64, depth=3, heads=4, drop=0.1,
                 use_pair_bias=True, pair_feat_idx=(0, 1, 2, 3), pool="attn"):
        super().__init__()
        self.use_pair_bias, self.pair_feat_idx, self.pool_kind = use_pair_bias, list(pair_feat_idx), pool
        self.embed = nn.Sequential(nn.Linear(in_feat, dim), nn.GELU(), nn.Linear(dim, dim))
        self.blocks = nn.ModuleList([EncoderBlock(dim, heads, drop=drop) for _ in range(depth)])
        self.pool = AttentionPooling(dim, heads) if pool == "attn" else None
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, n_classes))
        if use_pair_bias:
            self.pair_mlp = nn.Sequential(nn.Linear(len(self.pair_feat_idx), 16), nn.GELU(), nn.Linear(16, heads))
        self.last_attn, self.attn_maps = None, []
    def forward(self, x, mask, p4=None):
        h = self.embed(x); bias = None
        if self.use_pair_bias and p4 is not None:
            U = pairwise_features(p4)[..., self.pair_feat_idx]
            bias = self.pair_mlp(U).permute(0, 3, 1, 2)
        self.attn_maps = []
        for blk in self.blocks:
            h, self.last_attn = blk(h, mask, bias); self.attn_maps.append(self.last_attn)
        if self.pool_kind == "attn":
            z = self.pool(h, mask)
        else:
            z = (h * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1.0)   # masked mean
        return self.head(z)

def n_params(m): return sum(p.numel() for p in m.parameters())

In [ ]:
# --- load JetClass as masked sets: x (features), p4 (raw), pos (raw deta,dphi) -----------
import uproot, awkward as ak
CANDIDATE_PATHS = [
    "../data/JetClass_example_100k.root",
    "/Users/sanmay/Documents/ML_SCHOOLS/MLSCHOOL_2023_ICTS/Main_School/JetDataset/JetClass_example_100k.root",
    "./JetClass_example_100k.root",
]
root_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if root_path is None: raise FileNotFoundError("JetClass_example_100k.root not found")
print("Using:", root_path)

MAX_PART  = 128
N_PER_CLS = 2000     # per class; reduced from the module's 8000 so ~20 trainings stay fast
DIM, DEPTH, HEADS, EPOCHS = 64, 3, 4, 10

tree = uproot.open(root_path)["tree"]
br = tree.arrays(["part_px","part_py","part_pz","part_energy","part_deta","part_dphi",
                  "label_QCD","label_Tbqq","label_Tbl"])
is_qcd = ak.to_numpy(br["label_QCD"]).astype(bool)
is_top = ak.to_numpy(br["label_Tbqq"]).astype(bool) | ak.to_numpy(br["label_Tbl"]).astype(bool)
selidx = np.concatenate([np.where(is_qcd)[0][:N_PER_CLS], np.where(is_top)[0][:N_PER_CLS]])
labels = np.concatenate([np.zeros(N_PER_CLS), np.ones(N_PER_CLS)]).astype(np.int64)

px,py,pz,e = br["part_px"][selidx], br["part_py"][selidx], br["part_pz"][selidx], br["part_energy"][selidx]
deta,dphi  = br["part_deta"][selidx], br["part_dphi"][selidx]
pt = np.sqrt(px**2+py**2); dR = np.sqrt(deta**2+dphi**2)
sumpt, sume = ak.sum(pt,axis=1), ak.sum(e,axis=1)
FEATURE_NAMES = ["deta","dphi","log_pt","log_e","log_pt_rel","log_e_rel","dR"]
feat_list = [deta, dphi, np.log(pt+1e-8), np.log(e+1e-8),
             np.log(pt/sumpt+1e-8), np.log(e/sume+1e-8), dR]
def pad(f): return ak.to_numpy(ak.fill_none(ak.pad_none(f, MAX_PART, clip=True), 0.0))
X   = np.stack([pad(f) for f in feat_list], axis=-1).astype(np.float32)
P4  = np.stack([pad(px), pad(py), pad(pz), pad(e)], axis=-1).astype(np.float32)
POS = np.stack([pad(deta), pad(dphi)], axis=-1).astype(np.float32)                 # raw (eta,phi) for §5,§6
nparts = np.minimum(ak.to_numpy(ak.num(pt, axis=1)), MAX_PART)
MASK = (np.arange(MAX_PART)[None,:] < nparts[:,None]).astype(np.float32)

realmask = MASK.astype(bool); mean, std = X[realmask].mean(0), X[realmask].std(0)+1e-6
X = (X - mean) / std
Xt, P4t, POSt, Mt, Yt = map(torch.from_numpy, (X, P4, POS, MASK, labels))
g = torch.Generator().manual_seed(0); pm = torch.randperm(len(Yt), generator=g)
Xt, P4t, POSt, Mt, Yt = Xt[pm], P4t[pm], POSt[pm], Mt[pm], Yt[pm]
n = len(Yt); ntr, nva = int(0.70*n), int(0.15*n)
sl = {"train": slice(0,ntr), "val": slice(ntr,ntr+nva), "test": slice(ntr+nva,n)}

class JetSet(Dataset):
    def __init__(self, s): self.x, self.p4, self.m, self.y = Xt[s], P4t[s], Mt[s], Yt[s]
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.p4[i], self.m[i], self.y[i]
loaders = {k: DataLoader(JetSet(s), batch_size=64, shuffle=(k=="train")) for k, s in sl.items()}
print("jets:", {k: len(JetSet(s)) for k, s in sl.items()}, "| X", tuple(Xt.shape))

In [ ]:
# --- train / eval helpers (train_model stashes a per-epoch .history) ---------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps, ls, nt = [], [], 0.0, 0
    for x, p4, m, y in loader:
        x, p4, m, y = x.to(device), p4.to(device), m.to(device), y.to(device)
        logits = model(x, m, p4)
        ls += F.cross_entropy(logits, y, reduction="sum").item(); nt += y.size(0)
        ys.append(y.cpu()); ps.append(F.softmax(logits, -1)[:, 1].cpu())
    yv, pv = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return {"loss": ls/nt, "acc": ((pv>0.5).astype(int)==yv).mean(), "auc": roc_auc_score(yv, pv), "y": yv, "p": pv}

def background_rejection(y, p, eff=0.5):
    fpr, tpr, _ = roc_curve(y, p)
    eps_b = max(np.interp(eff, tpr, fpr), 1.0/max(int((y==0).sum()), 1))
    return 1.0/eps_b

def train_model(model, loaders, epochs=EPOCHS, lr=1e-3, wd=1e-4, quiet=True):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = {"train_loss": [], "val_loss": [], "val_auc": []}
    t0 = time.time()
    for ep in range(epochs):
        model.train(); run, nb = 0.0, 0
        for x, p4, m, y in loaders["train"]:
            x, p4, m, y = x.to(device), p4.to(device), m.to(device), y.to(device)
            loss = F.cross_entropy(model(x, m, p4), y)
            opt.zero_grad(); loss.backward(); opt.step(); run += loss.item(); nb += 1
        sched.step(); va = evaluate(model, loaders["val"])
        hist["train_loss"].append(run/max(nb,1)); hist["val_loss"].append(va["loss"]); hist["val_auc"].append(va["auc"])
        if not quiet: print(f"  epoch {ep+1:2d}: train {run/max(nb,1):.3f}  val AUC {va['auc']:.4f}")
    model.history = hist; model.train_secs = time.time()-t0
    return model

In [ ]:
# --- base models reused across exercises: a plain Transformer and a full ParT ------------
print("Base Plain Transformer (no pairwise bias):")
base_plain = train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=HEADS,
                                             use_pair_bias=False), loaders, quiet=False)
print("Base Particle Transformer (all 4 pairwise features):")
base_part = train_model(ParticleTransformer(len(FEATURE_NAMES), dim=DIM, depth=DEPTH, heads=HEADS,
                                            use_pair_bias=True), loaders, quiet=False)
res_plain, res_part = evaluate(base_plain, loaders["test"]), evaluate(base_part, loaders["test"])
print(f"\nBASELINES  Plain AUC {res_plain['auc']:.4f}   ParT AUC {res_part['auc']:.4f}")
print("Setup complete.")

## Exercise 1 — Ablate the bias features

> *Retrain ParT with the pairwise bias restricted to only ln Δ, or only ln m². Which of the four features
> carries most of ParT's advantage?*

**The concept.** ParT's edge over a plain Transformer is the pairwise bias `U_ij` added to the attention
logits, built from four QCD-splitting kinematics: `lnΔ` (angular separation), `ln k_T` (splitting hardness),
`ln z` (momentum sharing), `ln m²` (pair invariant mass). To measure each feature's *value* (§5a only
*visualized* them), we retrain ParT with the bias restricted to a **single** feature at a time and compare
test AUC to the full-4 ParT and the no-bias plain Transformer. The gap `AUC(feature) − AUC(plain)` is that
feature's standalone contribution.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 2 — Depth / heads / width

> *Sweep `depth`, `heads`, `dim`. Where are the diminishing returns for jets, and how does the training curve
> change?*

**The concept.** Three capacity knobs: **depth** (how many rounds of all-to-all message passing), **width**
`dim` (representation size), and **heads** (how many relationship types attended in parallel). More capacity
helps until the ~100-particle jet task saturates, after which you pay compute/overfitting for nothing. We
sweep each knob around the base config (depth 3, dim 64, heads 4), holding the others fixed, and read off AUC
vs parameters.

In [ ]:
def add(tag, model):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 3 — Class attention vs mean pool

> *Replace `AttentionPooling` with the masked mean-pool from Module 1. How much does learnable pooling
> actually help?*

**The concept.** The readout turns the set of particle tokens into one jet vector. Module 1 used a **masked
mean** (every particle equal); ParT uses **class attention** (PMA) — a learnable query that weights particles
by relevance. Our `ParticleTransformer(pool=...)` flips exactly this, keeping the encoder identical, so the
comparison isolates the pooling. We test it **with** the pairwise bias (full ParT) to see if learnable pooling
still helps once the encoder is already strong.

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 4 — ISAB accuracy trade-off

> *§10a benchmarked ISAB's speed. Now swap the encoder's full-attention blocks for `ISAB` (m=16, 32), retrain,
> and measure the AUC you trade for the linear scaling.*

**The concept.** Full self-attention (SAB) is `O(N²)`; the Set Transformer's **ISAB** routes all-to-all
attention through `m ≪ N` learnable *inducing points* (`X→I→X`), costing `O(Nm)` — the win for
high-multiplicity events. The question is what accuracy you pay. We build a Set-Transformer classifier whose
encoder is either full **SAB** or **ISAB(m)**, keep the PMA readout, and compare AUC at `m = 16, 32`. (ISAB
attends through inducing points, so it does not carry ParT's N×N pairwise bias — this is a pure
architecture-vs-cost comparison, run without the bias.)

In [ ]:
class MAB(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=2):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, A, B, key_mask=None):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
class SAB(nn.Module):
    def __init__(self, dim, heads):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, X, key_mask=None):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
class ISAB(nn.Module):
    def __init__(self, dim, heads, m=16):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, X, key_mask=None):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
class SetTransformerClf(nn.Module):
    """Encoder of SAB or ISAB blocks + PMA readout (no pairwise bias)."""
    def __init__(self, in_feat, dim=DIM, depth=DEPTH, heads=HEADS, block="sab", m=16):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, x, mask, p4=None):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 5 — Attention → graph: does it rediscover the k-NN?

> *Threshold a trained attention map at its top-k entries per row and compare the induced graph, jet-by-jet,
> to ParticleNet's k-NN graph. Does attention rediscover the same neighbours?*

**The concept.** §9a showed attention *enhances* small-ΔR pairs on average. Here we make it concrete and
per-jet: for each particle we take its **top-k most-attended** partners (from the base ParT's last block) and
compare that set to its **k nearest neighbours in (η, φ)** — exactly the graph ParticleNet's first block
builds. We report the mean overlap fraction against the random baseline `k/(n−1)`. High overlap = attention
learned, from data, the angular locality Module 2 imposed by hand.

In [ ]:
@torch.no_grad()
def attn_knn_overlap(model, k=8, nbatch=8):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 6 — Three-way race: PFN vs ParticleNet vs ParT

> *Put PFN (M1), ParticleNet (M2) and ParT (M3) on one ROC / rejection plot for the same test jets.*

**The concept.** The course's three archetypes on one axis: **no edges** (PFN / DeepSets), **hard k-NN edges**
(ParticleNet), **soft all-to-all edges** (ParT). We train a PFN (masked-sum DeepSets on the padded features)
and a ParticleNet (Module 2's PyG implementation, built from the **same** jet indices) and overlay their ROCs
with the base ParT — all evaluated on the identical test jets. Expect a clean ordering
**PFN < ParticleNet ≤ ParT**, with the biggest separation at high background rejection.

In [ ]:
# PFN over the padded set (masked-sum DeepSets), same loaders as ParT
class PFN(nn.Module):
    def __init__(self, in_feat, hidden=(64,128,128), latent=128):
        super().__init__()
        dims=[in_feat,*hidden]; layers=[]
        for a,b in zip(dims[:-1],dims[1:]): layers += [nn.Linear(a,b), nn.ReLU()]
        self.phi = nn.Sequential(*layers, nn.Linear(hidden[-1], latent))
        self.head = nn.Sequential(nn.Linear(latent,128), nn.ReLU(), nn.Linear(128,2))
    def forward(self, x, mask, p4=None):
        h = self.phi(x); z = (h*mask.unsqueeze(-1)).sum(1)
        return self.head(z)
torch.manual_seed(0); pfn = train_model(PFN(len(FEATURE_NAMES)), loaders); res_pfn = evaluate(pfn, loaders["test"])
print(f"PFN AUC {res_pfn['auc']:.4f}")

In [ ]:
# ParticleNet via PyG, built from the SAME shuffled indices -> identical test jets
import torch_geometric
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader

def knn_graph_native(x, k, batch):
    with torch.no_grad():
        N=x.size(0); x2=(x*x).sum(1,keepdim=True); d2=x2+x2.t()-2*(x@x.t())
        same=batch.unsqueeze(0)==batch.unsqueeze(1); d2=d2.masked_fill(~same,float("inf")); d2.fill_diagonal_(float("inf"))
        kk=min(k,max(N-1,1)); dk,idx=d2.topk(kk,dim=1,largest=False); valid=torch.isfinite(dk)
        c=torch.arange(N,device=x.device).unsqueeze(1).expand(-1,kk)
        return torch.stack([idx[valid], c[valid]], 0)
class EdgeConv(MessagePassing):
    def __init__(self, ic, oc, aggr="mean"):
        super().__init__(aggr=aggr)
        self.mlp=nn.Sequential(nn.Linear(2*ic,oc),nn.BatchNorm1d(oc),nn.ReLU(),nn.Linear(oc,oc),nn.BatchNorm1d(oc),nn.ReLU())
    def forward(self,x,ei): return self.propagate(ei,x=x)
    def message(self,x_i,x_j): return self.mlp(torch.cat([x_i,x_j-x_i],-1))
class Block(nn.Module):
    def __init__(self,ic,oc): super().__init__(); self.c=EdgeConv(ic,oc); self.s=nn.Linear(ic,oc)
    def forward(self,x,ei): return F.relu(self.c(x,ei)+self.s(x))
class ParticleNet(nn.Module):
    def __init__(self, inf, k=8, ch=(32,64,64), fc=128):
        super().__init__(); self.k=k; chs=[inf,*ch]
        self.blocks=nn.ModuleList([Block(chs[i],chs[i+1]) for i in range(len(ch))])
        self.head=nn.Sequential(nn.Linear(ch[-1],fc),nn.ReLU(),nn.Dropout(0.1),nn.Linear(fc,2))
    def forward(self,data):
        batch=data.batch; feats,coords=data.x,data.pos
        for blk in self.blocks:
            ei=knn_graph_native(coords,self.k,batch); feats=blk(feats,ei); coords=feats
        return self.head(global_mean_pool(feats,batch))

def make_geo(split):
    s=sl[split]; xs,ps,ms,ys=Xt[s],POSt[s],Mt[s],Yt[s]; dl=[]
    for i in range(len(ys)):
        nn_=int(ms[i].sum())
        if nn_<2: continue
        dl.append(Data(x=xs[i,:nn_], pos=ps[i,:nn_], y=ys[i].view(1)))
    return dl
geo = {k: GeoLoader(make_geo(k), batch_size=64, shuffle=(k=="train")) for k in ("train","val","test")}

def train_pnet(model, epochs=EPOCHS):
    model=model.to(device); opt=torch.optim.Adam(model.parameters(),1e-3)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    for ep in range(epochs):
        model.train()
        for d in geo["train"]:
            d=d.to(device); loss=F.cross_entropy(model(d),d.y); opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model
@torch.no_grad()
def eval_pnet(model):
    model.eval(); ys,ps=[],[]
    for d in geo["test"]:
        d=d.to(device); ps.append(F.softmax(model(d),-1)[:,1].cpu()); ys.append(d.y.cpu())
    y,p=torch.cat(ys).numpy(),torch.cat(ps).numpy()
    return {"y":y,"p":p,"auc":roc_auc_score(y,p)}
torch.manual_seed(0); pnet=train_pnet(ParticleNet(len(FEATURE_NAMES),k=8)); res_pnet=eval_pnet(pnet)
print(f"ParticleNet AUC {res_pnet['auc']:.4f}")

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 7 — Boost augmentation

> *The bias is longitudinal-boost invariant (§5a) but the input features (log_pt, log_e, …) are not. Add random
> longitudinal boosts as data augmentation: does it help, and does ParT — already partly boost-aware through
> its bias — benefit less than the plain Transformer?*

**The concept.** A longitudinal (beam-direction) boost by rapidity ξ leaves `pT` and φ invariant but shifts
rapidity by ξ and changes `E` (`E' = E cosh ξ + pz sinh ξ`). So the *input* features that use energy/pseudo-
rapidity (`log_e`, `log_e_rel`, `deta`, `dR`) **move under a boost**, while ParT's pairwise bias — built from
rapidity *differences*, `pT`, and `m²` — is **invariant**. We therefore expect: (i) a boosted test set hurts
models trained only at rest; (ii) boost **augmentation** recovers robustness; (iii) ParT, already partly
boost-aware, degrades **less** and needs augmentation **less** than the plain Transformer. We featurize
directly from `p4` (self-consistently) so boosting is exact.

In [ ]:
def featurize_p4(p4, mask, stats=None):
    """(B,N,4) raw 4-vectors -> (B,N,7) standardized features, matching FEATURE_NAMES, computed FROM p4."""
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def boost_z(p4, xi):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def train_boost(use_bias, augment, xi_max=2.0, epochs=8):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def auc_at_boost(model, xi):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Exercise 8 — Faithful attribution: gradient-weighted rollout

> *§9e showed attention and gradient saliency only partly agree. Implement gradient-weighted attention rollout
> (Chefer et al., 2021): before composing, weight each layer's attention by
> E_h[(∂y_c/∂A_ℓ) ⊙ A_ℓ]₊. Does the importance map shift onto the particles the gradient says matter?*

**The concept.** Plain rollout (§9d) composes the raw per-layer attention `Ā_ℓ` — but it is *class-agnostic*
(it ignores which class you are explaining). **Chefer et al.** make it class-specific by weighting each layer
by its **gradient w.r.t. the target logit**: `Ā_ℓ ← (∇_{A_ℓ} y_c ⊙ A_ℓ)₊`, head-averaged, before composing
with the residual-identity and folding in the (grad-weighted) class-attention readout. The prediction is that
this map aligns **better with gradient saliency** than plain rollout, because it is built from the gradient.
We measure the top-k overlap with input-gradient saliency for both.

In [ ]:
def _norm(imp, m):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def class_importance(model, x, p4, m):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def rollout_importance(model, x, p4, m):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def grad_rollout_importance(model, x, p4, m, cls=1):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def grad_saliency(model, x, p4, m):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
def overlap_at_k(a, b, m, k):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

In [ ]:
# TODO (exercise): implement this cell — see the prompt above.
#   Reuse the models and helpers defined in Setup.
raise NotImplementedError

## Wrap-up

You've reached the end of the worksheet. If every exercise runs without a `NotImplementedError`, you've implemented them all — compare your results and reasoning with the solutions notebook. The through-line across the course: **every model is constrained message passing** — a choice of relations, a permutation-invariant aggregation, and a baked-in symmetry.